In [1]:
import pandas as pd
import os

# Update these paths to match your actual Drive or local folders
# file_id = '1FtRdaCpGFbUYhKbnp-TBJQeBRFgn5y3v'
csv_path = r'G:\My Drive\df_1_to_25.csv'
video_folder = r'G:\My Drive\ASL_Videos_100'

# Load the CSV into a Pandas DataFrame
df = pd.read_csv(csv_path, dtype = {'gloss': str, 'video_id': str, 'frame_start': int, 'frame_end': int, 'fps': int })

# Print the first few rows to ensure frame_start, frame_end, gloss, and video_id are loaded
print(df.head())

      gloss video_id  frame_start  frame_end  fps
0  accident    00618          374        400   25
1  accident    00624            1         -1   25
2  accident    00626            1         -1   25
3  accident    00629            1         -1   25
4  accident    00634            1         -1   25


In [10]:
import pandas as pd
import os
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- LOCAL CONFIGURATION ---
# Use relative paths or absolute paths matching your local machine
csv_path = 'top_100_filtered.csv' 
video_folder = './ASL_Videos_100'

# Load the CSV into a Pandas DataFrame
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path, dtype={
        'gloss': str, 
        'video_id': str, 
        'frame_start': int, 
        'frame_end': int, 
        'fps': int 
    })
    print(f"Loaded {len(df)} rows from {csv_path}")
else:
    print(f"ERROR: CSV file not found at {csv_path}")

# State management
to_delete = set()

def build_advanced_review_gui():
    global df

    main_output = widgets.Output()
    search_bar = widgets.Text(
        placeholder='Type gloss (e.g. APPLE) and press Enter...',
        description='Search:',
        continuous_update=False
    )

    status_label = widgets.Label(value=f"Total entries: {len(df)} | Folder: {video_folder}")

    def on_delete_toggled(change):
        vid_id = str(change['owner'].description.replace("Delete: ", "")).strip()
        if change['new']:
            to_delete.add(vid_id)
        else:
            to_delete.discard(vid_id)

    def execute_cleanup(b):
        global df
        with main_output:
            if not to_delete:
                print("⚠️ No videos selected.")
                return

            print(f"🚀 Processing deletion of {len(to_delete)} items...")
            ids_to_remove = list(to_delete)

            for vid_id in ids_to_remove:
                v_filename = f"{vid_id}.mp4"
                v_path = os.path.join(video_folder, v_filename)

                # Local File Removal
                if os.path.exists(v_path):
                    try:
                        os.remove(v_path)
                    except Exception as e:
                        print(f"Error deleting {v_filename}: {e}")

                # Update DataFrame
                df = df[df['video_id'].astype(str) != vid_id]

            # Save Locally
            try:
                df.to_csv(csv_path, index=False)
                print(f"✅ CSV Updated Locally: {csv_path}")
            except Exception as e:
                print(f"❌ Save Error: {e}")

            to_delete.clear()
            update_display(search_bar.value)

    def update_display(search_query):
        search_query = search_query.strip().upper()
        with main_output:
            clear_output()

            if not search_query:
                print("Enter a gloss to load videos.")
                return

            mask = df['gloss'].str.upper().str.contains(search_query, na=False)
            filtered_results = df[mask]

            if filtered_results.empty:
                print(f"No results for: '{search_query}'")
                return

            found_glosses = sorted(filtered_results['gloss'].unique())

            for gloss in found_glosses:
                gloss_subset = filtered_results[filtered_results['gloss'] == gloss]
                grid_items = []

                for _, row in gloss_subset.iterrows():
                    vid = str(row['video_id']).strip()
                    v_filename = f"{vid}.mp4"
                    v_path = os.path.join(video_folder, v_filename)

                    meta = widgets.HTML(f"<div style='font-size:11px;'>ID: {vid}<br>{row['frame_start']}-{row['frame_end']}</div>")
                    chk = widgets.Checkbox(description=f"Delete: {vid}", value=(vid in to_delete), indent=False)
                    chk.observe(on_delete_toggled, names='value')

                    if os.path.exists(v_path):
                        # Ensure the path is accessible to the browser
                        v_preview = widgets.Video.from_file(v_path, width=160)
                    else:
                        v_preview = widgets.HTML("<div style='width:160px;height:100px;background:#eee;'>Missing</div>")

                    card = widgets.VBox([v_preview, meta, chk], layout=widgets.Layout(padding='5px', border='1px solid #ddd', margin='2px'))
                    grid_items.append(card)

                grid = widgets.GridBox(grid_items, layout=widgets.Layout(grid_template_columns="repeat(auto-fill, 180px)"))
                display(widgets.HTML(f"<b>{gloss}</b> ({len(gloss_subset)} videos)"))
                display(grid)

    cleanup_btn = widgets.Button(description="Apply Changes", button_style='danger')
    cleanup_btn.on_click(execute_cleanup)
    search_bar.observe(lambda change: update_display(change['new']), names='value')

    ui = widgets.VBox([
        widgets.HTML("<h2>Local ASL Curator</h2>"),
        widgets.HBox([search_bar, cleanup_btn]),
        status_label,
        main_output
    ])
    display(ui)

build_advanced_review_gui()

ERROR: CSV file not found at top_100_filtered.csv
